# 🔍 LangSmith Observability — Your First Traced LLM Call

**Goal:** send a single question to an OpenAI chat model and watch the whole call show up as a *trace* in [LangSmith](https://smith.langchain.com).

### In a nutshell
This notebook loads your OpenAI and LangSmith API keys from a `.env` file (secure key management — no secrets in the code) and verifies that tracing is enabled without exposing those secrets. It then initializes an OpenAI chat model via LangChain with deterministic output (`temperature=0`) and defines a reusable prompt template for concise answers. A function decorated with `@traceable` sends formatted questions to the model and logs each run in LangSmith for observability. Finally, when executed, it asks a sample question about the benefits of observability in AI agents, retrieves the model's response, and prints both the input and output — demonstrating how to integrate OpenAI with LangChain and LangSmith while keeping keys safe and calls fully traceable.

### Why observability?
When you build with LLMs, a lot happens that you can't see from the return value alone: the exact prompt that was sent, how long the model took, how many tokens it used, what it cost, and whether anything errored. **Observability** means capturing all of that automatically so you can debug, measure, and improve your app. LangSmith is LangChain's tool for exactly this.

### The pipeline
```
   your question (str)
          │
          ▼
   ┌────────────────────┐
   │   PromptTemplate   │   "Answer clearly and concisely:\n{question}"
   └────────────────────┘
          │  formatted prompt
          ▼
   ┌────────────────────┐
   │    HumanMessage    │
   └────────────────────┘
          │  messages
          ▼
   ┌────────────────────┐        ┌────────────────────────┐
   │     ChatOpenAI     │ ─────▶ │   OpenAI Chat API      │
   │    llm.invoke()    │ ◀───── │   (e.g. gpt-5-mini)    │
   └────────────────────┘        └────────────────────────┘
          │  response.content (str)
          ▼
      printed answer

   ══ the whole call is wrapped by @traceable("SimpleAgentTrace") ══▶  📊 LangSmith
```

Each box becomes a run you can inspect in LangSmith once you reach the end of the notebook.

## Step 1 — Install the dependencies

Run this once per environment. `langchain-openai` gives us `ChatOpenAI`, and `python-dotenv` lets us keep secrets in a `.env` file instead of hard-coding them. After it finishes, **restart the kernel** so the new packages load cleanly.

In [ ]:
%pip install langchain langchain-openai langchain-community python-dotenv

## Step 2 — Load secrets and switch tracing on

`load_dotenv()` reads a local `.env` file into environment variables, so your keys never sit in the notebook itself (secure key management). For this notebook your `.env` should define:

| Variable | What it's for |
|---|---|
| `LANGSMITH_API_KEY` | Authenticates you to LangSmith (create one at smith.langchain.com → Settings). |
| `OPENAI_API_KEY` | Authenticates you to OpenAI. |
| `OpenAI_MODEL_NAME` | Which model to call, e.g. `gpt-5-mini`. |

Setting `LANGSMITH_TRACING="true"` is the switch that turns tracing on; `LANGSMITH_PROJECT` names the folder your traces land in. The sanity check prints only *whether* the key exists — never the key itself.

In [ ]:
import os
from dotenv import load_dotenv

from langsmith import traceable
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage

load_dotenv()  # Load environment variables from a local .env file if present

# ---------- Turn LangSmith tracing ON ----------
# With these set, every LangChain/LangSmith call in this kernel is recorded
# and shipped to your LangSmith project automatically.
os.environ["LANGSMITH_TRACING"] = "true"
os.environ.setdefault("LANGSMITH_PROJECT", "OpenAI-LLM-Demo")

# Sanity check (never print the secret itself, only whether it exists)
print(
    "Tracing on?:", os.getenv("LANGSMITH_TRACING") == "true",
    "| Project:", os.getenv("LANGSMITH_PROJECT"),
    "| Has LangSmith key?:", bool(os.getenv("LANGSMITH_API_KEY"))
)

## Step 3 — Build the chain: model → prompt → traced function

Three small pieces:
- **`ChatOpenAI`** — the model client. We set `temperature=0` for deterministic output.
- **`PromptTemplate`** — a reusable prompt with a `{question}` slot.
- **`simple_agent_response`** — our function, decorated with **`@traceable`**. That decorator is the key idea: it turns this function into a named run in LangSmith, and the `llm.invoke()` inside it is captured as a nested child run — so you see the *structure* of your call, not just the output.

In [ ]:
# ---------- The chat model ----------
# temperature=0 makes the output deterministic (great for demos and comparing traces).
llm = ChatOpenAI(
    model_name=os.environ["OpenAI_MODEL_NAME"],  # e.g. "gpt-5-mini", read from your .env
    temperature=0,
)

# ---------- The prompt template ----------
# A reusable prompt with a {question} slot we fill in at call time.
prompt = PromptTemplate.from_template(
    "Answer clearly and concisely:\n{question}"
)

# ---------- The traceable function ----------
# @traceable makes this function appear as a named run in LangSmith.
# The llm.invoke() call inside it is captured as a nested child run automatically.
@traceable(name="SimpleAgentTrace")
def simple_agent_response(question: str) -> str:
    formatted_prompt = prompt.format(question=question)
    response = llm.invoke([HumanMessage(content=formatted_prompt)])
    return response.content

## Step 4 — Run it and generate a trace

Running the cell below calls the model and prints the answer. Behind the scenes it also sends a complete trace to LangSmith. Try changing `q` and re-running — each run becomes its own trace you can compare side by side.

In [ ]:
# ---------- Run it — this is what fires the trace to LangSmith ----------
if __name__ == "__main__":
    q = "What are the key benefits of observability in AI agents?"
    print("Question:", q)
    answer = simple_agent_response(q)
    print("\nAnswer:\n", answer)

## 📊 What to observe in LangSmith

Open [smith.langchain.com](https://smith.langchain.com), then go to **Projects → `OpenAI-LLM-Demo`** (the project name you set in Step 2). Each time you ran the cell above, a new trace appeared. Click the most recent one — named **`SimpleAgentTrace`** — and look for:

- **The run tree (nesting).** A parent run `SimpleAgentTrace` (your `@traceable` function) with a child `ChatOpenAI` run nested inside it. This mirrors the pipeline diagram at the top of the notebook.
- **Inputs.** The `question` string passed to your function, and the fully-formatted prompt that was actually sent to the model.
- **Outputs.** The model's answer text (`response.content`).
- **Token usage.** Prompt tokens, completion tokens, and total — useful for estimating cost.
- **Latency.** Total wall-clock time plus a per-step breakdown, so you can see where the time went.
- **Cost.** An estimated dollar cost (when pricing is known for the model).
- **Model & parameters.** The model name and settings (temperature, etc.) used for the call.
- **Status.** Green ✅ for success, red ❌ for errors — and if it failed, the exception and stack trace are captured right there. This is where observability really earns its keep.
- **Metadata & timestamps.** Run IDs, start/end times, and any tags — handy for filtering later.

**Try this:** run the notebook 2–3 times with different questions, then compare the traces' latency and token counts in the project table.

### Key takeaways
| Concept | Takeaway |
|---|---|
| Tracing switch | `LANGSMITH_TRACING="true"` + a `LANGSMITH_API_KEY` is all it takes to start capturing runs. |
| `@traceable` | Turns any function into a named, nested run you can inspect in LangSmith. |
| Observability | Lets you see the prompt, tokens, latency, cost, and errors — not just the final answer. |
| Projects | `LANGSMITH_PROJECT` groups related runs so you can compare and filter them. |